# CIFAR-10 avec le framework from scratch

Ici, on utilise la logique actuelle du projet :

- `Dense`
- `ReLU`
- `Softmax`
- `MulticlassCrossEntropy`
- `Sequential`
- `SGD`

Le dataset change, mais la logique d'apprentissage reste celle du framework maison.


**But** : monter Drive et definir un espace de travail persistant.

**Pourquoi** : garder le dataset, les checkpoints et les sorties entre les sessions Colab.


In [22]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/learning-cifar10-from-scratch')
DATA_ROOT = PROJECT_ROOT / 'data'
MODEL_ROOT = PROJECT_ROOT / 'models'

for path in [PROJECT_ROOT, DATA_ROOT, MODEL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


PosixPath('/content/drive/MyDrive/learning-cifar10-from-scratch')

**But** : installer juste les dependances utiles au notebook.

**Pourquoi** : charger CIFAR-10 et reutiliser le code local du framework.


In [23]:
!pip -q install torchvision matplotlib

**But** : charger le notebook, les outils numeriques et le framework maison.

**Pourquoi** : garantir qu'on benchmarke notre propre logique, pas un modele externe.


In [25]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets

REPO_ROOT = Path('/content/drive/MyDrive/formations/learning')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.activations import ReLU, Softmax
from src.layers import Dense
from src.losses import MulticlassCrossEntropy
from src.models import Sequential
from src.optim import SGD
from src.training.sequential import train_multiclass_classifier

np.set_printoptions(precision=4, suppress=True)


ImportError: cannot import name 'Softmax' from 'src.activations' (/content/scratch-deep-learning-vision/src/activations/__init__.py)

**But** : charger CIFAR-10 et transformer les images en vecteurs.

**Pourquoi** : notre framework actuel sait traiter des couches `Dense`, donc on a besoin d'entrees aplaties.


In [ ]:
train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)
test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)
class_names = train_dataset.classes
class_names

**But** : voir quelques exemples du dataset.

**Pourquoi** : garder un lien visuel avec les classes avant de passer aux vecteurs d'entree.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, index in zip(axes.ravel(), range(10)):
    image, label = train_dataset[index]
    ax.imshow(image)
    ax.set_title(class_names[label])
    ax.axis('off')
plt.tight_layout()
plt.show()


**But** : preparer un sous-ensemble raisonnable, normaliser et encoder les labels en one-hot.

**Pourquoi** : CIFAR-10 complet est lourd pour un MLP from scratch pur `numpy`, surtout dans Colab.


In [ ]:
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 2000
NUM_CLASSES = 10
SEED = 42

rng = np.random.default_rng(SEED)

train_indices = rng.choice(len(train_dataset), size=TRAIN_SAMPLES, replace=False)
test_indices = rng.choice(len(test_dataset), size=TEST_SAMPLES, replace=False)

def to_numpy_subset(dataset, indices):
    images = []
    labels = []
    for index in indices:
        image, label = dataset[index]
        images.append(np.asarray(image, dtype=np.float32) / 255.0)
        labels.append(label)
    X = np.stack(images)
    y = np.array(labels, dtype=int)
    return X, y

X_train_img, y_train_idx = to_numpy_subset(train_dataset, train_indices)
X_test_img, y_test_idx = to_numpy_subset(test_dataset, test_indices)

X_train = X_train_img.reshape(len(X_train_img), -1)
X_test = X_test_img.reshape(len(X_test_img), -1)

mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

y_train = np.eye(NUM_CLASSES)[y_train_idx]
y_test = np.eye(NUM_CLASSES)[y_test_idx]

X_train.shape, y_train.shape, X_test.shape, y_test.shape

**But** : definir un MLP multiclasses avec notre framework.

**Pourquoi** : c'est la version la plus fidele a l'etat actuel du projet avant d'avoir des convolutions from scratch.


In [ ]:
model = Sequential([
    Dense(input_dim=32 * 32 * 3, output_dim=256, seed=0),
    ReLU(),
    Dense(input_dim=256, output_dim=128, seed=1),
    ReLU(),
    Dense(input_dim=128, output_dim=NUM_CLASSES, seed=2),
    Softmax(),
])

loss = MulticlassCrossEntropy()
optimizer = SGD(lr=0.01)
model

**But** : entrainer le modele sur train et suivre la performance sur test.

**Pourquoi** : mesurer ce que notre framework sait deja faire sur un vrai benchmark d'images.


In [ ]:
history = train_multiclass_classifier(
    model=model,
    X=X_train,
    y=y_train,
    loss=loss,
    optimizer=optimizer,
    epochs=30,
    batch_size=128,
    shuffle=True,
    seed=SEED,
)

history[-1]

**But** : visualiser la loss et l'accuracy train.

**Pourquoi** : verifier la convergence du MLP avant l'evaluation finale sur le test set.


In [ ]:
epochs = [row.epoch for row in history]
losses = [row.loss for row in history]
accuracies = [row.accuracy for row in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, losses)
axes[0].set_title('Train loss')
axes[0].set_xlabel('epoch')
axes[0].grid(True, alpha=0.25)

axes[1].plot(epochs, accuracies)
axes[1].set_title('Train accuracy')
axes[1].set_xlabel('epoch')
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


**But** : evaluer le modele sur le test set.

**Pourquoi** : mesurer la generalisation reelle du MLP from scratch sur CIFAR-10.


In [ ]:
from src.metrics import multiclass_accuracy

test_probs = model.predict_proba(X_test)
test_acc = multiclass_accuracy(y_test, test_probs)
test_pred_idx = np.argmax(test_probs, axis=1)
test_true_idx = np.argmax(y_test, axis=1)

print('test accuracy =', round(test_acc, 4))

**But** : afficher la matrice de confusion.

**Pourquoi** : voir quelles classes le MLP confond le plus.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(test_true_idx, test_pred_idx)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.show()


## Suite logique

- ajouter la regularisation
- tester des profondeurs plus grandes
- ajouter `Conv2D` from scratch
- comparer MLP vs CNN from scratch sur le meme benchmark
